<a href="https://colab.research.google.com/github/mggg/Training_Materials/blob/main/notebooks/technical/Tech_1_math_of_votekit_live.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -q votekit

<b><font color="#bb0000"> Restart the runtime after the above cell runs for the first time</font></b>

# Math of VoteKit

This notebook is a collection of live demos of VoteKit. This is intended to focus on the underlying math of VoteKit, and should be used as a companion to the "Math of VoteKit" presentation.

# Spatial models




In [ ]:
from votekit.ballot_generator import spacial_profile_and_positions_generator
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Choose number of voters n
# And the number of candidates m
n = 100
m = 5
candidates = [str(i) for i in range(m)]

# We can use any numpy distribution (or custom distribution -- more on that later)
# to randomly sample  positions for voters and canidates.
# Here we sample from the following distributions distributions
# Voters: Normal(mean = 1/2, std = 1/10) in 2d
# Candidates: Uniform(0,1) in 2d

# Define a dictionary of parameters for both distributions
# for a full list of possible distributions and their
# required parameters check out:
# https://numpy.org/doc/1.16/reference/routines.random.html
voter_params = {"loc": 0.5, "scale": 0.1, "size": 2}
candidate_params = {"low": 0, "high": 1, "size": 2}

# We also define a distance function to compute the
# distances between any pair of voters and candidates.
# Here, we just use euclidean distance
distance = lambda point1, point2: np.linalg.norm(point1 - point2)

# Generate a profile from random candidate and voter positions
profile, candidate_position_dict, voter_positions = spacial_profile_and_positions_generator(
    number_of_ballots=n,
    candidates=candidates,
    voter_dist=np.random.normal,
    voter_dist_kwargs=voter_params,
    candidate_dist=np.random.uniform,
    candidate_dist_kwargs=candidate_params,
    distance=distance,
)

In [ ]:
# And then visualize the results
candidate_positions = np.array([i for i in candidate_position_dict.values()])
pal = sns.color_palette("hls", 8)
plt.scatter(voter_positions[:, 0], voter_positions[:, 1], label="voters", color=pal[4])
plt.scatter(
    candidate_positions[:, 0],
    candidate_positions[:, 1],
    label="candidates",
    color=pal[1],
)
plt.legend()

# IC, IAC


In [ ]:
from votekit.plots import multi_profile_fpv_plot, profile_fpv_plot
import votekit.ballot_generator as bg

# generate a profile to work with first
candidates = ["A", "B", "C"]

# initializing the ballot generator
profile1 = bg.ic_profile_generator(candidates=candidates, number_of_ballots=1000)


print("IC profile:")
print(profile1.df)

fig1 = profile_fpv_plot(profile1, title="First Place Votes in Profile 1", candidate_ordering=["A", "B", "C"])

In [ ]:
profile2 = bg.iac_profile_generator(candidates=candidates, number_of_ballots=1000)

print("IC profile:")
print(profile2.df)

fig2 = profile_fpv_plot(profile2, title="First Place Votes in Profile 1", candidate_ordering=["A", "B", "C"])



# dirichlet for Pref Intervals


In [ ]:
from votekit.pref_interval import PreferenceInterval

strong_pref_interval = PreferenceInterval.from_dirichlet(
    candidates=["A", "B", "C"], alpha=0.1
)
print("Strong preference for one candidate", {c: f"{x:.2f}" for c,x in strong_pref_interval.interval.items()})

abo_pref_interval = PreferenceInterval.from_dirichlet(
    candidates=["A", "B", "C"], alpha=1
)
print("All bets are off preference", {c: f"{x:.2f}" for c,x in abo_pref_interval.interval.items()})

unif_pref_interval = PreferenceInterval.from_dirichlet(
    candidates=["A", "B", "C"], alpha=10
)
print("Uniform preference for all candidates", {c: f"{x:.2f}" for c,x in unif_pref_interval.interval.items()})


# PL, BT


In [ ]:
import votekit.ballot_generator as bg
from votekit.ballot_generator import BlocSlateConfig
from votekit import PreferenceInterval

# the sPL model assumes there are blocs of voters,
# but we can just say that there is only one bloc
bloc_voter_prop = {"all_voters": 1}
slate_to_candidates = {"all_voters": ["A", "B", "C"]}

# the preference interval (80,15,5)
pref_intervals_by_bloc = {
    "all_voters": {"all_voters": PreferenceInterval({"A": 0.80, "B": 0.15, "C": 0.05})}
}

# the sPL model needs an estimate of cohesion between blocs,
# but there is only one bloc here
cohesion_parameters = {"all_voters": {"all_voters": 1}}

config = BlocSlateConfig(
    n_voters=100,
    bloc_proportions=bloc_voter_prop,
    slate_to_candidates=slate_to_candidates,
    preference_mapping=pref_intervals_by_bloc,
    cohesion_mapping=cohesion_parameters,
)


profile = bg.slate_pl_profile_generator(config)
print(profile.df)

In [ ]:
slate_to_candidates = {"Alpha": ["A", "B"], "Xenon": ["X", "Y"]}

# note that we include candidates with 0 support,
# and that our preference intervals will automatically rescale to sum to 1

pref_intervals_by_bloc = {
    "Alpha": {
        "Alpha": PreferenceInterval({"A": 0.8, "B": 0.2}),
        "Xenon": PreferenceInterval({"X": 0.00001,"Y": 0.99999}),
    },
    "Xenon": {
        "Alpha": PreferenceInterval({"A": 0.5, "B": 0.5}),
        "Xenon": PreferenceInterval({"X": 0.5, "Y": 0.5}),
    },
}


bloc_voter_prop = {"Alpha": 0.8, "Xenon": 0.2}

# assume that each bloc is 90% cohesive
# we'll discuss exactly what that means later
cohesion_parameters = {
    "Alpha": {"Alpha": 0.9, "Xenon": 0.1},
    "Xenon": {"Xenon": 0.9, "Alpha": 0.1},
}

config = BlocSlateConfig(
    n_voters=10000,
    bloc_proportions=bloc_voter_prop,
    slate_to_candidates=slate_to_candidates,
    preference_mapping=pref_intervals_by_bloc,
    cohesion_mapping=cohesion_parameters,
)

# the by_bloc parameter allows us to see which ballots came from which blocs of voters
profile_dict = bg.slate_bt_profiles_by_bloc_generator(config)
print("The ballots from Alpha voters\n", profile_dict["Alpha"].df)

print("The ballots from Xenon voters\n", profile_dict["Xenon"].df)

profile_list = list(profile_dict.values())
agg_profile = sum(profile_list[1:], start=profile_list[0])
print("Aggregated ballots\n", agg_profile.df)


# CS


In [ ]:
bloc_voter_prop = {"W": 0.8, "C": 0.2}

# the values of .9 indicate that these blocs are highly polarized;
# they prefer their own candidates much more than the opposing slate
cohesion_parameters = {"W": {"W": 0.9, "C": 0.1}, "C": {"C": 0.9, "W": 0.1}}

alphas = {"W": {"W": 2, "C": 1}, "C": {"W": 1, "C": 0.5}}

slate_to_candidates = {"W": ["W1", "W2", "W3"], "C": ["C1", "C2"]}

config = BlocSlateConfig(
    n_voters=1000,
    bloc_proportions=bloc_voter_prop,
    slate_to_candidates=slate_to_candidates,
    cohesion_mapping=cohesion_parameters,
)
config.set_dirichlet_alphas(alphas=alphas)


profile = bg.cambridge_profile_generator(config)
print(profile.df.head(10).to_string())